In [3]:
import os
import glob
import numpy as np
import torch
import torch.nn as nn
from torch.nn import Sequential as Seq, Linear as Lin, ReLU, BatchNorm1d as BN
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, f1_score, jaccard_score
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import open3d as o3d

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [4]:
DATA_DIR = "../dataset"
NUM_POINTS = 4096
BATCH_SIZE = 4
EPOCHS = 10
LEARNING_RATE = 0.001
TRAIN_RATIO = 0.7
VAL_RATIO = 0.15
TEST_RATIO = 0.15
USE_CUDA = torch.cuda.is_available()
DEVICE = torch.device("cuda" if USE_CUDA else "cpu")
MAX_FILES = 500

# Dataset

In [5]:
def read_ply_ascii(file_path):
    """Read ASCII .ply with x, y, z, scalar_Label."""
    with open(file_path, 'r') as f:
        lines = f.readlines()
    header_end = 0
    for i, line in enumerate(lines):
        if line.strip() == 'end_header':
            header_end = i + 1
            break
    data_lines = lines[header_end:]
    points = []
    for line in data_lines:
        parts = line.strip().split()
        if len(parts) == 4:
            x, y, z, label = map(float, parts)
            points.append([x, y, z, int(label)])
    points = np.array(points, dtype=np.float32)
    coords = points[:, :3]
    labels = points[:, 3].astype(np.int64)
    return coords, labels

def normalize_point_cloud(coords):
    """Center and scale into unit sphere."""
    centroid = np.mean(coords, axis=0)
    coords_centered = coords - centroid
    max_dist = np.max(np.linalg.norm(coords_centered, axis=1))
    if max_dist > 0:
        coords_normalized = coords_centered / max_dist
    else:
        coords_normalized = coords_centered
    return coords_normalized, centroid, max_dist

def augment_point_cloud(coords, labels):
    """Augment: rotation, noise, scaling, dropout."""
    theta = np.random.uniform(0, 2 * np.pi)
    rot_mat = np.array([[np.cos(theta), -np.sin(theta), 0],
                        [np.sin(theta),  np.cos(theta), 0],
                        [0, 0, 1]])
    coords_aug = coords @ rot_mat.T
    noise = np.random.normal(0, 0.01, size=coords_aug.shape)
    coords_aug += noise
    scale = np.random.uniform(0.8, 1.2)
    coords_aug *= scale
    if np.random.rand() < 0.5:
        keep_mask = np.random.rand(len(coords_aug)) > 0.05
        coords_aug = coords_aug[keep_mask]
        labels = labels[keep_mask]
    return coords_aug, labels

class LidarValveDataset(Dataset):
    def __init__(self, file_list, normalize=True, augment=False, num_points=NUM_POINTS):
        self.file_list = file_list
        self.normalize = normalize
        self.augment = augment
        self.num_points = num_points

    def __len__(self):
        return len(self.file_list)

    def __getitem__(self, idx):
        coords, labels = read_ply_ascii(self.file_list[idx])
        if self.normalize:
            coords, _, _ = normalize_point_cloud(coords)
        if self.augment:
            coords, labels = augment_point_cloud(coords, labels)
        if len(coords) >= self.num_points:
            choice = np.random.choice(len(coords), self.num_points, replace=False)
        else:
            choice = np.random.choice(len(coords), self.num_points, replace=True)
        coords = coords[choice]
        labels = labels[choice]
        coords = torch.tensor(coords, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.long)
        return coords, labels

# PointNet

In [6]:
class PointNetSeg(nn.Module):
    def __init__(self, num_classes, input_dim=3):
        super().__init__()
        self.local_mlp = nn.Sequential(
            nn.Conv1d(input_dim, 64, 1),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.Conv1d(64, 128, 1),
            nn.BatchNorm1d(128),
            nn.ReLU()
        )
        self.global_mlp = nn.Sequential(
            nn.Conv1d(128, 1024, 1),
            nn.BatchNorm1d(1024),
            nn.ReLU()
        )
        self.mlp2 = nn.Sequential(
            nn.Conv1d(128 + 1024, 512, 1),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Conv1d(512, 256, 1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, num_classes, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        local_feat = self.local_mlp(x)
        pre_global = self.global_mlp(local_feat)
        global_feat = torch.max(pre_global, dim=2, keepdim=True)[0]
        global_feat = global_feat.repeat(1, 1, local_feat.shape[2])
        concat = torch.cat([local_feat, global_feat], dim=1)
        logits = self.mlp2(concat)
        return logits.transpose(1, 2)


# PointNet++

In [7]:
def farthest_point_sample(xyz, npoint):
    """
    Input:
        xyz: point cloud data, [B, N, 3]
        npoint: number of samples
    Return:
        centroids: sampled point indices, [B, npoint]
    """
    B, N, C = xyz.shape
    device = xyz.device
    centroids = torch.zeros(B, npoint, dtype=torch.long).to(device)
    distance = torch.ones(B, N).to(device) * 1e10
    farthest = torch.randint(0, N, (B,), dtype=torch.long).to(device)
    batch_indices = torch.arange(B, dtype=torch.long).to(device)
    for i in range(npoint):
        centroids[:, i] = farthest
        centroid = xyz[batch_indices, farthest, :].view(B, 1, 3)
        dist = torch.sum((xyz - centroid) ** 2, dim=-1)  # [B, N]
        mask = dist < distance
        distance[mask] = dist[mask]
        farthest = torch.max(distance, dim=-1)[1]
    return centroids

def index_points(points, idx):
    """
    Input:
        points: [B, N, C]
        idx: [B, npoint] or [B, npoint, nsample]
    Return:
        new_points: [B, npoint, C] or [B, npoint, nsample, C]
    """
    device = points.device
    B = points.shape[0]
    view_shape = list(idx.shape)
    view_shape[1:] = [1] * (len(view_shape) - 1)
    repeat_shape = list(idx.shape)
    repeat_shape[0] = 1
    batch_indices = torch.arange(B, dtype=torch.long).to(device).view(view_shape).repeat(repeat_shape)
    new_points = points[batch_indices, idx, :]
    return new_points

def square_distance(src, dst):
    """
    Calculate squared distance between src and dst.
    src: [B, N, C]
    dst: [B, M, C]
    return: [B, N, M]
    """
    B, N, C = src.shape
    _, M, _ = dst.shape
    dist = -2 * torch.matmul(src, dst.permute(0, 2, 1))
    dist += torch.sum(src ** 2, dim=-1).view(B, N, 1)
    dist += torch.sum(dst ** 2, dim=-1).view(B, 1, M)
    return dist

def ball_query(radius, nsample, xyz, new_xyz):
    """
    Input:
        radius: local region radius
        nsample: max number of points in each region
        xyz: all points, [B, N, 3]
        new_xyz: query points, [B, npoint, 3]
    Return:
        idx: [B, npoint, nsample] indices of neighbors
    """
    B, N, C = xyz.shape
    npoint = new_xyz.shape[1]
    dist = square_distance(new_xyz, xyz)  # [B, npoint, N]
    idx = dist.argsort(dim=-1)[:, :, :nsample]  # [B, npoint, nsample]
    dist_to_neighbors = torch.gather(dist, dim=-1, index=idx)
    mask = dist_to_neighbors > radius ** 2
    idx[mask] = 0
    return idx

class PointNetSetAbstraction(nn.Module):
    def __init__(self, npoint, radius, nsample, in_channel, mlp, group_all=False):
        super().__init__()
        self.npoint = npoint
        self.radius = radius
        self.nsample = nsample
        self.group_all = group_all
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv2d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm2d(out_channel))
            last_channel = out_channel

    def forward(self, xyz, points):
        B, N, C = xyz.shape
        if self.group_all:
            new_xyz = torch.mean(xyz, dim=1, keepdim=True)
            if points is not None:
                new_points = points.permute(0, 2, 1).unsqueeze(-1)
            else:
                new_points = xyz.permute(0, 2, 1).unsqueeze(-1)
            grouped_points = new_points.repeat(1, 1, 1, N)
        else:
            fps_idx = farthest_point_sample(xyz, self.npoint)
            new_xyz = index_points(xyz, fps_idx)
            idx = ball_query(self.radius, self.nsample, xyz, new_xyz)
            grouped_xyz = index_points(xyz, idx)
            grouped_xyz_norm = grouped_xyz - new_xyz.view(B, self.npoint, 1, 3)
            if points is not None:
                grouped_points = index_points(points, idx)
                new_points = torch.cat([grouped_xyz_norm, grouped_points], dim=-1)
            else:
                new_points = grouped_xyz_norm
            new_points = new_points.permute(0, 3, 2, 1)  # [B, C+3, nsample, npoint]

        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = conv(new_points)
            new_points = bn(new_points)
            new_points = torch.relu(new_points)
        new_points = torch.max(new_points, dim=2)[0]
        new_points = new_points.permute(0, 2, 1)
        return new_xyz, new_points

class PointNetFeaturePropagation(nn.Module):
    def __init__(self, in_channel, mlp):
        super().__init__()
        self.mlp_convs = nn.ModuleList()
        self.mlp_bns = nn.ModuleList()
        last_channel = in_channel
        for out_channel in mlp:
            self.mlp_convs.append(nn.Conv1d(last_channel, out_channel, 1))
            self.mlp_bns.append(nn.BatchNorm1d(out_channel))
            last_channel = out_channel

    def forward(self, xyz1, xyz2, points1, points2):
        B, N1, _ = xyz1.shape
        _, N2, _ = xyz2.shape
        if N2 == 1:
            interpolated = points2.repeat(1, N1, 1)
        else:
            dist = square_distance(xyz1, xyz2)
            dist, idx = torch.sort(dist, dim=-1)
            dist, idx = dist[:, :, :3], idx[:, :, :3]
            dist_recip = 1.0 / (dist + 1e-8)
            norm = torch.sum(dist_recip, dim=-1, keepdim=True)
            weight = dist_recip / norm
            interpolated = torch.sum(index_points(points2, idx) * weight.unsqueeze(-1), dim=2)
        if points1 is not None:
            new_points = torch.cat([interpolated, points1], dim=-1)
        else:
            new_points = interpolated
        new_points = new_points.permute(0, 2, 1)
        for i, conv in enumerate(self.mlp_convs):
            bn = self.mlp_bns[i]
            new_points = conv(new_points)
            new_points = bn(new_points)
            new_points = torch.relu(new_points)
        new_points = new_points.permute(0, 2, 1)
        return new_points

class PointNetPlusPlusSeg(nn.Module):
    def __init__(self, num_classes, input_dim=3):
        super().__init__()
        self.sa1 = PointNetSetAbstraction(1024, 0.2, 32, input_dim, [64, 64, 128], group_all=False)
        self.sa2 = PointNetSetAbstraction(256, 0.4, 32, 128 + 3, [128, 128, 256], group_all=False)
        self.sa3 = PointNetSetAbstraction(64, 0.8, 32, 256 + 3, [256, 512, 1024], group_all=False)
        self.fp3 = PointNetFeaturePropagation(1024 + 256, [256, 256])
        self.fp2 = PointNetFeaturePropagation(256 + 128, [256, 128])
        self.fp1 = PointNetFeaturePropagation(128, [128, 128, num_classes])

    def forward(self, xyz):
        l1_xyz, l1_points = self.sa1(xyz, None)
        l2_xyz, l2_points = self.sa2(l1_xyz, l1_points)
        l3_xyz, l3_points = self.sa3(l2_xyz, l2_points)
        l2_points = self.fp3(l2_xyz, l3_xyz, l2_points, l3_points)
        l1_points = self.fp2(l1_xyz, l2_xyz, l1_points, l2_points)
        l0_points = self.fp1(xyz, l1_xyz, None, l1_points)
        return l0_points

# DGCNN

In [8]:
def knn(x, k):
    inner = -2 * torch.matmul(x.transpose(2, 1), x)
    xx = torch.sum(x ** 2, dim=1, keepdim=True)
    pairwise_distance = -xx - inner - xx.transpose(2, 1)
    idx = pairwise_distance.topk(k=k, dim=-1)[1]
    return idx

class EdgeConv(nn.Module):
    def __init__(self, in_channels, out_channels, k=20):
        super().__init__()
        self.k = k
        self.mlp = nn.Sequential(
            nn.Conv2d(in_channels*2, out_channels, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU()
        )

    def forward(self, x):
        B, C, N = x.shape
        idx = knn(x, self.k)
        idx_base = torch.arange(0, B, device=x.device).view(-1, 1, 1) * N
        idx = idx + idx_base
        idx = idx.view(-1)
        x_flat = x.transpose(2, 1).contiguous().view(-1, C)
        neighbors = x_flat[idx].view(B, N, self.k, C)
        x_expand = x.transpose(2, 1).unsqueeze(2).expand(B, N, self.k, C)
        edge = neighbors - x_expand
        edge = torch.cat([x_expand, edge], dim=-1).permute(0, 3, 1, 2)
        out = self.mlp(edge)
        out = torch.max(out, dim=-1)[0]
        return out

class DGCNNSeg(nn.Module):
    def __init__(self, num_classes, k=20):
        super().__init__()
        self.k = k
        self.conv1 = EdgeConv(3, 64, k)
        self.conv2 = EdgeConv(64, 64, k)
        self.conv3 = EdgeConv(64, 64, k)
        self.conv4 = EdgeConv(64, 128, k)
        self.mlp = nn.Sequential(
            nn.Conv1d(64+64+64+128, 256, 1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Conv1d(256, 256, 1),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Conv1d(256, num_classes, 1)
        )

    def forward(self, x):
        x = x.transpose(1, 2)
        x1 = self.conv1(x)
        x2 = self.conv2(x1)
        x3 = self.conv3(x2)
        x4 = self.conv4(x3)
        x_concat = torch.cat([x1, x2, x3, x4], dim=1)
        logits = self.mlp(x_concat)
        return logits.transpose(1, 2)


# Training

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, device, desc=""):
    model.train()
    total_loss = 0
    loop = tqdm(loader, desc=f"{desc} Train", leave=False)
    for coords, labels in loop:
        coords, labels = coords.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(coords)
        loss = criterion(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        loop.set_postfix(loss=loss.item())
    return total_loss / len(loader)

def evaluate(model, loader, criterion, device, num_classes, desc=""):
    model.eval()
    total_loss = 0
    all_preds = []
    all_labels = []
    with torch.no_grad():
        loop = tqdm(loader, desc=f"{desc} Eval", leave=False)
        for coords, labels in loop:
            coords, labels = coords.to(device), labels.to(device)
            logits = model(coords)
            loss = criterion(logits.reshape(-1, logits.shape[-1]), labels.reshape(-1))
            total_loss += loss.item()
            preds = torch.argmax(logits, dim=-1)
            all_preds.append(preds.cpu().numpy().ravel())
            all_labels.append(labels.cpu().numpy().ravel())
    all_preds = np.concatenate(all_preds)
    all_labels = np.concatenate(all_labels)
    overall_acc = accuracy_score(all_labels, all_preds)
    iou_per_class = jaccard_score(all_labels, all_preds, labels=range(num_classes), average=None, zero_division=0)
    miou = np.mean(iou_per_class)
    f1 = f1_score(all_labels, all_preds, average='weighted')
    conf_matrix = confusion_matrix(all_labels, all_preds, labels=range(num_classes))
    return overall_acc, miou, iou_per_class, f1, conf_matrix, total_loss/len(loader)

def plot_confusion_matrix(cm, class_names, save_path):
    plt.figure(figsize=(10, 8))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=class_names, yticklabels=class_names)
    plt.xlabel('Predicted')
    plt.ylabel('True')
    plt.title('Confusion Matrix')
    plt.tight_layout()
    plt.savefig(save_path)
    plt.close()

def plot_training_curves(train_losses, val_losses, val_mious, model_name):
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
    ax1.plot(train_losses, label='Train Loss')
    ax1.plot(val_losses, label='Val Loss')
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Loss')
    ax1.legend()
    ax1.set_title(f'{model_name} - Loss')
    ax2.plot(val_mious, label='Val mIoU', color='green')
    ax2.set_xlabel('Epoch')
    ax2.set_ylabel('mIoU')
    ax2.legend()
    ax2.set_title(f'{model_name} - mIoU')
    plt.savefig(f'{model_name}_curves.png')
    plt.close()

def train_and_evaluate(model, name, train_loader, val_loader, test_loader, num_classes):
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=EPOCHS//2, gamma=0.5)
    criterion = nn.CrossEntropyLoss()

    train_losses, val_losses, val_mious = [], [], []
    best_miou = -1.0
    best_state = None

    for epoch in range(1, EPOCHS+1):
        print(f"\nEpoch {epoch}/{EPOCHS}")
        train_loss = train_one_epoch(model, train_loader, optimizer, criterion, DEVICE, desc=name)
        val_acc, val_miou, _, _, _, val_loss = evaluate(model, val_loader, criterion, DEVICE, num_classes, desc=name)
        scheduler.step()

        train_losses.append(train_loss)
        val_losses.append(val_loss)
        val_mious.append(val_miou)

        print(f"  Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val mIoU: {val_miou:.4f} | Val Acc: {val_acc:.4f}")

        if val_miou > best_miou:
            best_miou = val_miou
            best_state = model.state_dict().copy()

    if best_state is not None:
        model.load_state_dict(best_state)
    else:
        print(f"Warning: No improvement for {name}, using last model state.")

    test_acc, test_miou, test_iou, test_f1, test_cm, _ = evaluate(model, test_loader, criterion, DEVICE, num_classes, desc=name)
    print(f"\nTest results for {name}:")
    print(f"  OA: {test_acc:.4f}, mIoU: {test_miou:.4f}, F1: {test_f1:.4f}")
    print(f"  Per-class IoU: {np.array2string(test_iou, precision=4)}")

    plot_training_curves(train_losses, val_losses, val_mious, name)
    class_names = [f"Class_{i}" for i in range(num_classes)]
    plot_confusion_matrix(test_cm, class_names, f"{name}_confusion.png")
    print(f"  Plots saved for {name}")

    return {
        "OA": test_acc,
        "mIoU": test_miou,
        "F1": test_f1,
        "Per-class IoU": test_iou,
        "Confusion Matrix": test_cm,
        "Train Losses": train_losses,
        "Val Losses": val_losses,
        "Val mIoUs": val_mious
    }

In [10]:
all_files = sorted(glob.glob(os.path.join(DATA_DIR, "*.ply")))
if len(all_files) == 0:
    raise FileNotFoundError(f"No .ply files found in {DATA_DIR}")
print(f"Found {len(all_files)} files. Taking first {MAX_FILES}.")
all_files = all_files[:MAX_FILES]

max_label = 0
for f in all_files[:10]:
    _, labels = read_ply_ascii(f)
    max_label = max(max_label, labels.max())
num_classes = max_label + 1
print(f"Number of classes: {num_classes}")

train_files, temp_files = train_test_split(all_files, train_size=TRAIN_RATIO, random_state=42)
val_files, test_files = train_test_split(temp_files, train_size=VAL_RATIO/(VAL_RATIO+TEST_RATIO), random_state=42)
print(f"Train: {len(train_files)}, Val: {len(val_files)}, Test: {len(test_files)}")

train_dataset = LidarValveDataset(train_files, normalize=True, augment=True, num_points=NUM_POINTS)
val_dataset = LidarValveDataset(val_files, normalize=True, augment=False, num_points=NUM_POINTS)
test_dataset = LidarValveDataset(test_files, normalize=True, augment=False, num_points=NUM_POINTS)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

Found 500 files. Taking first 500.
Number of classes: 10
Train: 350, Val: 75, Test: 75


In [11]:
model = PointNetSeg(num_classes).to(DEVICE)
results_pointnet = train_and_evaluate(model, "PointNet", train_loader, val_loader, test_loader, num_classes)


Epoch 1/10


  Train Loss: 1.2746 | Val Loss: 1.3389 | Val mIoU: 0.2275 | Val Acc: 0.5229

Epoch 2/10


  Train Loss: 0.8371 | Val Loss: 0.7446 | Val mIoU: 0.3928 | Val Acc: 0.7118

Epoch 3/10


  Train Loss: 0.6606 | Val Loss: 0.6189 | Val mIoU: 0.4243 | Val Acc: 0.7559

Epoch 4/10


  Train Loss: 0.5964 | Val Loss: 0.3993 | Val mIoU: 0.5112 | Val Acc: 0.8453

Epoch 5/10


...

  Train Loss: 0.3762 | Val Loss: 0.2894 | Val mIoU: 0.5776 | Val Acc: 0.8952

Epoch 8/10


  Train Loss: 0.3482 | Val Loss: 0.3260 | Val mIoU: 0.5510 | Val Acc: 0.8683

Epoch 9/10


  Train Loss: 0.3281 | Val Loss: 0.5525 | Val mIoU: 0.4966 | Val Acc: 0.7819

Epoch 10/10


  Train Loss: 0.3380 | Val Loss: 0.2793 | Val mIoU: 0.5968 | Val Acc: 0.8845



Test results for PointNet:
  OA: 0.8898, mIoU: 0.5982, F1: 0.8853
  Per-class IoU: [0.7173 0.7951 0.86   0.7015 0.0174 0.2616 0.818  0.927  0.     0.8845]
  Plots saved for PointNet


In [12]:
model = PointNetPlusPlusSeg(num_classes).to(DEVICE)
results_pointnetpp = train_and_evaluate(model, "PointNet++", train_loader, val_loader, test_loader, num_classes)


Epoch 1/10


  Train Loss: 1.2710 | Val Loss: 1.0876 | Val mIoU: 0.5065 | Val Acc: 0.7942

Epoch 2/10


  Train Loss: 0.8405 | Val Loss: 1.0321 | Val mIoU: 0.5401 | Val Acc: 0.7937

Epoch 3/10


  Train Loss: 0.7288 | Val Loss: 0.9553 | Val mIoU: 0.5445 | Val Acc: 0.8042

Epoch 4/10


  Train Loss: 0.6327 | Val Loss: 0.9586 | Val mIoU: 0.5849 | Val Acc: 0.8035

Epoch 5/10


...

  Train Loss: 0.4666 | Val Loss: 0.7191 | Val mIoU: 0.5997 | Val Acc: 0.8488

Epoch 8/10


  Train Loss: 0.4463 | Val Loss: 0.7420 | Val mIoU: 0.6140 | Val Acc: 0.8384

Epoch 9/10


  Train Loss: 0.4305 | Val Loss: 0.7281 | Val mIoU: 0.6387 | Val Acc: 0.8438

Epoch 10/10


  Train Loss: 0.4062 | Val Loss: 0.5619 | Val mIoU: 0.6797 | Val Acc: 0.8816



Test results for PointNet++:
  OA: 0.8930, mIoU: 0.6944, F1: 0.8893
  Per-class IoU: [0.7041 0.6966 0.8389 0.8683 0.5025 0.6189 0.8056 0.9313 0.1132 0.865 ]
  Plots saved for PointNet++


In [13]:
model = DGCNNSeg(num_classes).to(DEVICE)
results_dgcnn = train_and_evaluate(model, "DGCNN", train_loader, val_loader, test_loader, num_classes)


Epoch 1/10


  Train Loss: 1.3599 | Val Loss: 1.1017 | Val mIoU: 0.2219 | Val Acc: 0.5656

Epoch 2/10


  Train Loss: 1.0466 | Val Loss: 0.8202 | Val mIoU: 0.3501 | Val Acc: 0.6827

Epoch 3/10


  Train Loss: 0.8318 | Val Loss: 0.6830 | Val mIoU: 0.3991 | Val Acc: 0.7198

Epoch 4/10


  Train Loss: 0.6793 | Val Loss: 0.4869 | Val mIoU: 0.5770 | Val Acc: 0.8098

Epoch 5/10


...

  Train Loss: 0.4892 | Val Loss: 0.4335 | Val mIoU: 0.6175 | Val Acc: 0.8187

Epoch 8/10


  Train Loss: 0.4661 | Val Loss: 0.3712 | Val mIoU: 0.6617 | Val Acc: 0.8512

Epoch 9/10


  Train Loss: 0.4515 | Val Loss: 0.5323 | Val mIoU: 0.5720 | Val Acc: 0.7694

Epoch 10/10


  Train Loss: 0.4391 | Val Loss: 0.4359 | Val mIoU: 0.6207 | Val Acc: 0.8236



Test results for DGCNN:
  OA: 0.8222, mIoU: 0.6160, F1: 0.8233
  Per-class IoU: [0.535  0.5465 0.5098 0.5499 0.2881 0.5717 0.8007 0.9084 0.5874 0.8627]
  Plots saved for DGCNN


In [14]:
print("\n" + "="*60)
print("MODEL COMPARISON TABLE")
print("="*60)
print(f"{'Model':<15} {'OA':<8} {'mIoU':<8} {'F1':<8}")
for name, res in [("PointNet", results_pointnet), ("PointNet++", results_pointnetpp), ("DGCNN", results_dgcnn)]:
    print(f"{name:<15} {res['OA']:.4f}   {res['mIoU']:.4f}   {res['F1']:.4f}")


MODEL COMPARISON TABLE
Model           OA       mIoU     F1      
PointNet        0.8898   0.5982   0.8853
PointNet++      0.8930   0.6944   0.8893
DGCNN           0.8222   0.6160   0.8233


In [ ]:
def visualize_prediction_open3d(model, dataset, device, idx=0):
    model.eval()
    coords, true_labels = dataset[idx] 
    coords_batch = coords.unsqueeze(0).to(device)
    with torch.no_grad():
        logits = model(coords_batch)
        preds = torch.argmax(logits, dim=-1).squeeze(0).cpu().numpy()
    coords_np = coords.numpy()
    true_labels_np = true_labels.numpy()

    unique_labels = np.unique(np.concatenate([true_labels_np, preds]))
    n_classes = len(unique_labels)
    cmap = plt.cm.get_cmap('tab20', n_classes)
    colors_true = np.array([cmap(label % n_classes)[:3] for label in true_labels_np])
    colors_pred = np.array([cmap(label % n_classes)[:3] for label in preds])

    pcd_true = o3d.geometry.PointCloud()
    pcd_true.points = o3d.utility.Vector3dVector(coords_np)
    pcd_true.colors = o3d.utility.Vector3dVector(colors_true)

    pcd_pred = o3d.geometry.PointCloud()
    pcd_pred.points = o3d.utility.Vector3dVector(coords_np)
    pcd_pred.colors = o3d.utility.Vector3dVector(colors_pred)

    o3d.visualization.draw_geometries([pcd_true], window_name="Input", width=800, height=600)
    o3d.visualization.draw_geometries([pcd_pred], window_name="Output", width=800, height=600)

visualize_prediction_open3d(model, test_dataset, DEVICE, idx=0)

C:\Users\user\AppData\Local\Temp\ipykernel_1108\2945783235.py:13: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed in 3.11. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap()`` or ``pyplot.get_cmap()`` instead.
  cmap = plt.cm.get_cmap('tab20', n_classes)
